In [1]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import networkx as nx
import random
import heapq
import collections
from copy import deepcopy, copy

### All the simulation uses hours and kilometers

In [2]:
ARRIVAL_RATE = np.array([314.2, 162.4, 138.6, 148.8, 273.2, 1118.8, 2773.8, 4036.2, 4237.4, 3277.0, 2843.0, 2876.4, 3143.0, 3277.8, 3546.2, 4335.0, 4945.4, 4525.8, 2847.8, 1828.0, 1378.4, 1271.2, 1171.2, 767.6])
INCIDENT_RATE = np.array([0.1935483870967742, 0.25806451612903225, 0.2903225806451613, 0.1935483870967742, 0.06451612903225806, 0.967741935483871, 32.16129032258065, 131.03225806451613, 157.2258064516129, 48.32258064516129, 8.451612903225806, 6.258064516129032, 8.161290322580646, 10.290322580645162, 19.0, 65.7741935483871, 157.41935483870967, 163.3548387096774, 36.54838709677419, 3.7096774193548385, 1.2258064516129032, 0.8709677419354839, 0.5806451612903226, 0.41935483870967744])
INCIDENT_DURATION_PARAMS = (1.2294425495153518, 0.09878316410683122, 5.082687545787541)
# INCIDENT_DURATION_PARAMS = (1.2294425495153518, 0.09878316410683122, 5.082687545787541)

incident_duration_dist = stats.lognorm(*INCIDENT_DURATION_PARAMS)

MEAN_ACCIDENT_DURATION = 1 

In [3]:
#Sampling arrival times of cars to network
def lambdat(lambda_t, t : np.array):
    lambdat = []
    for time in t:
        lambdat.append(lambda_t[int(np.floor(time))])
    return lambdat

def arrival_times(lam): #Taken from lecture notes
    max_T = 24
    arrival_times = collections.deque()
    exp_dist = stats.expon(scale = 1/lam)
    t = exp_dist.rvs()
    while t < max_T:
        arrival_times.append(t)
        t += exp_dist.rvs()
    
    return np.asarray(arrival_times)

In [4]:
class FES:
    def __init__(self):
        self.events = []

    def add(self, event):
        heapq.heappush(self.events, event)
    
    def next(self):
        return heapq.heappop(self.events)
    
    def isEmpty(self):
        return len(self.events) == 0
    
    def __repr__(self):
        string = ''
        sorted_events = sorted(self.events)
        for event in sorted_events:
            string += f'{event}\n'
        return string

In [5]:
class Queue:
    def __init__(self, road):
        self.road = road
        self.cars = []

    def add(self, car, time):
        heapq.heappush(self.cars, car)
        car.enter_queue(time)

    def add_list(self, list, time):
        for car in list:
            heapq.heappush(self.cars, car)
            car.enter_queue(time)   

    def next(self):
        return heapq.heappop(self.cars)
    
    def first(self):
        return heapq.nsmallest(1, self.cars)
    
    def isEmpty(self):
        return len(self.cars) == 0
    
    def __repr__(self):
        string = ''
        sorted_cars = sorted(self.cars)
        for car in sorted_cars:
            string += f'{car}\n'
        return string

In [6]:
class Event:
    TYPE = ['New car', 'Car departure', 'Accident', 'Accident end', 'left queue']
    def __init__(self, typ:int, time, car = None, road = None, duration=None, location=None):
        #types:
            #0 : Arrival of car to the network
            #1 : Car leaves current road and goes on to the next
            #2 : Accident in road
            #3 : End of accident
            #4 : Departure of car from queue of accident
        self.type = typ
        self.time = time
        self.road = road
        self.location = location
        self.duration = duration
        self.cancelled = False

        if typ == 0:
            car = Car(time_entrance = time)
    
        self.car = car
        
    def __str__(self):
        if self.type == 0:
            return f'{self.TYPE[self.type]} from {self.car.origin} to {self.car.destination} at {self.time}'
        if self.type == 1:
            return f'{self.TYPE[self.type]} of {self.car} at {self.time}h'
        if self.type == 2:
            return f'{self.TYPE[self.type]} at {self.road} at {self.time}h'
        if self.type == 3:
            return f'{self.TYPE[self.type]} at {self.road} at {self.time}h'
        if self.type == 4:
            return f'{self.car} {self.TYPE[self.type]} at {self.road} at {self.time}h'

    def __lt__(self, other):
        return self.time < other.time
    
    def new_time(self, new_time):
        self.time = new_time

In [ ]:
class Car:
    VELOCITIES = [100, 80]
    VELOCITIES_P = [0.9, 0.1]
    NAVIGATION_P = 0.5
    def __init__(self, time_entrance, origin = None, destination = None):
        #Origin and destination
        origin, destination = np.random.choice(JUNCTIONS, 2, replace = False)
        
        self.origin = origin
        self.destination = destination

        #path to follow
        self.path = nx.shortest_path(Graph, self.origin, self.destination, weight = 'length')

        #Velocity
        self.velocity = np.random.choice(self.VELOCITIES, p=self.VELOCITIES_P)

        #Variable to keep track how far into the path we are (to simplify scheduling events)
        #Int between 0 and len(path) - 1 that indicates in which edge we are, starting at 0
        #Essentially, how many edges has it travelled so far
        self.progress = 0
        
        #Give it nav with 10% chance
        self.has_nav = np.random.choice([True, False], p=[self.NAVIGATION_P, 1 - self.NAVIGATION_P])
        self.changed_route = []

        #Time entrance
        self.time = time_entrance
        self.time_entrance = time_entrance

        #keeping track of amount of accidents encountered
        self.accidents = 0
        self.next_event = None



    def __str__(self):
        return f'Vehicle travelling from {self.origin} to {self.destination} at {self.velocity} km/h, atm at {self.path[self.progress-1],self.path[self.progress]}, time {self.time}h'
    

    def custom_weight(self,u,v,data):
        """
        :param u: std for accepting function as weight, node 1
        :param v:  std for accepting function as weight, node 2
        :param data: std for accepting function as weight, edge
        :return: time_to_travel + delay
        """
        length = data['length']
        time_to_travel = (length / (self.velocity /3.6)) / 3600 #Mean of normal dist
        if Graph.edges[(u,v)]['accident']:   
            delay = MEAN_ACCIDENT_DURATION
        else:
            delay = 0
        return time_to_travel + delay
    
    def calc_time_to_travel(self,length):
        """
        :param length: edge length
        :return: returns time to traverse length based on normal speed
        """
        mean = length / (self.velocity /3.6) #seconds
        std = mean / 20
        time_to_travel = np.random.normal(loc = mean, scale = std) / 3600 #back to hours
        return max(0,time_to_travel)

    def schedule_event_exit(self):
        #Remove car from list of cars in previous edge
        if self.progress > 0:
            edge = (self.path[self.progress - 1], self.path[self.progress])
            DIC_EDGES[edge].remove(self)


        if self.progress < len(self.path) - 1:
            # Get the current and next node on the original path
            current_node = self.path[self.progress]
            next_node = self.path[self.progress + 1]
            edge = Graph.edges[(current_node, next_node)]

            # Check for accident on the current edge and if nav is enabled
            # No need to check for accident, just check if navigation (there may be accidents later that makes us recompute)
            if self.has_nav:
                # print("Updating path")
                new_path = nx.shortest_path(Graph, current_node, self.destination, weight=self.custom_weight)

                if new_path != self.path[self.progress:]:
                    # print("Found a quicker route")
                    # print(f"Prev path = {[Graph.nodes[node]['name'] for node in self.path]}")
                    # print(f"New path = {[Graph.nodes[node]['name'] for node in new_path]}")

                    #I commented this out bc append is not efficient
                    # self.changed_route.append((self.path, new_path))

                    # Preserve the already traversed portion
                    prefix = self.path[:self.progress]

                    updated_path = prefix + new_path
                    # print(updated_path)
                    # self.path = updated_path
                    # print(f"Updated path: {[Graph.nodes[node]['name'] for node in self.path]}")

                    # Update next_node based on the new path
                    next_node = self.path[self.progress + 1]
                    edge = Graph.edges[(current_node, next_node)]
                    length = edge['length']
                    # Sample travel time along the new edge (including accident delay)
                    
                    #I think no need to add accident duration bc i account for this in my code alr and a accident can happen before scheduling this
                    # time_to_travel = self.calc_time_to_travel(length) + edge['accident_duration']
                    time_to_travel = self.calc_time_to_travel(length)
                else:
                    # No change in path (fallback scenario)
                    length = edge['length']
                    time_to_travel = self.calc_time_to_travel(length)
            else:
                # Normal travel (no accident on the edge)
                length = edge['length']
                time_to_travel = self.calc_time_to_travel(length)

            new_time = self.time + time_to_travel
            # Schedule next event and update car's progress and time
            self.next_event = Event(1, new_time, car=self)
            # , road=(current_node, next_node))
            self.increase_progress()
            self.increase_time(new_time)

            #Add car to list of cars in the new edge
            # edge = (self.path[self.progress - 1], self.path[self.progress])
            DIC_EDGES[(current_node, next_node)].append(self)

            return self.next_event
    
    def increase_progress(self):
        self.progress += 1

    def increase_time(self, new_time):
        self.time = new_time

    def __lt__(self, other):
        return self.time < other.time
    
    def enter_queue(self, time):
        #store time at which the car entered the queue
        self.time_enter_q = time
        self.next_event.cancelled = True
        self.accidents += 1

    def exit_queue(self, time_exit):
        #Update time of car when it left the queue
        self.time += time_exit - self.time_enter_q
        # print(self.progress)
        new_event_exit = copy(self.next_event)
        # print(self.progress)
        new_event_exit.new_time(self.time)
        new_event_exit.cancelled = False
        # self.next_event.cancelled = True
        self.next_event = new_event_exit
        return new_event_exit

In [39]:
#Using a thining approach to get arrival times
max_lambda = np.max(ARRIVAL_RATE) + 1
all_arrivals = arrival_times(max_lambda)

uniform_dist = stats.uniform(0,1)
u_rvs = uniform_dist.rvs(len(all_arrivals))
accept_filter = u_rvs * max_lambda < lambdat(ARRIVAL_RATE, all_arrivals)

accepted_arrivals = all_arrivals[accept_filter]

In [40]:
#Using a thining approach to get accident times
max_lambda = np.max(INCIDENT_RATE) + 1
all_accidents = arrival_times(max_lambda)

uniform_dist = stats.uniform(0,1)
u_rvs = uniform_dist.rvs(len(all_accidents))
accept_filter = u_rvs * max_lambda < lambdat(INCIDENT_RATE, all_accidents)

accepted_accidents = all_accidents[accept_filter]

In [66]:
Graph = nx.read_gml('./data/networkAssignment.gml')
Graph = Graph.to_directed()
JUNCTIONS = list(Graph.nodes)
DIC_EDGES = {}
#Dictionary that keeps track on when the last accident ends in a road before another one can take place
DIC_ACCIDENTS = {}
for edge in Graph.edges:
    DIC_EDGES[edge] = []
    DIC_ACCIDENTS[edge] = 0
    #Associate departure from queue distribution to each edge based on number of lanes
    rate = 0.2/(60 * Graph.edges[edge]['lanes'])
    Graph.edges[edge]['DepDist'] = stats.expon(scale = rate)

    Graph.edges[edge]['accident'] = False
    Graph.edges[edge]['queue'] = False
    # Graph.edges[edge]['accident_duration'] = 0

In [67]:
#Simulation (can be turned into an object later)
LIST_CARS = []
LIST_AMOUNT_CARS_TRAFFIC_CHANGE = [0]
TIME_ARRAY = []
fes = FES()
for arrival in accepted_arrivals:
    #Two events associated with each arrival
    arrival_event = Event(0, arrival)
    fes.add(arrival_event)

for accident in accepted_accidents:
    affected_road = list(Graph.edges)[np.random.choice(len(Graph.edges))]
    if DIC_ACCIDENTS[affected_road] < accident:
        location = stats.uniform.rvs()
        duration = incident_duration_dist.rvs()/60
        time_end = accident + duration
        accident_event = Event(2, accident, road=affected_road, duration=duration, location=location)
        end_accident = Event(3, time_end, road=affected_road)

        fes.add(accident_event)
        fes.add(end_accident)
        DIC_ACCIDENTS[affected_road] = time_end
# heapq.heapify(fes.events)

In [68]:
# road = list(DIC_EDGES.keys())[0]
# accident = Event(2, 0.5, road=road, duration=0.1, location=0.4)
# end_accident = Event(3, 0.6, road =road)
# fes.add(accident)
# fes.add(end_accident)
# # # road = road[::-1]
# # # accident = Event(2, 0.5, road=road)
# # # end_accident = Event(3, 0.6, road =road)
# # # fes.add(accident)
# # # fes.add(end_accident)

In [69]:
#Queue for accidents setup
Queues = {}
for edge in DIC_EDGES.keys():
    Queues[edge] = Queue(edge)

In [ ]:
t = 0 #current time
step = 0
while t < 24.0:
    event = fes.next()
    if not(event.cancelled):
        if t> event.time:
            print('OH OH OH OH')
            print(event)
        t = event.time
        TIME_ARRAY.append(t)
        
        
        # print(t)
        #Car joins network
        if event.type == 0:
            
            car = event.car
            
            #Schedule next event and store it as attribute
            car.next_event = car.schedule_event_exit()
            car_travel_event = car.next_event
            if car_travel_event.time <= t:
                print('Problem 0')
            fes.add(car_travel_event)
            LIST_CARS.append(event.car)
            LIST_AMOUNT_CARS_TRAFFIC_CHANGE.append(0)

        #Car leaves road
        if event.type == 1:
            
            car = event.car
            # print(car.progress)
            # print(car in DIC_EDGES[(car.path[car.progress - 1], car.path[car.progress])])
            # print(car)
            next_travel_event_car = car.schedule_event_exit()
            if type(next_travel_event_car) == Event: #If the car has arrived to its destination it wont return an event object
                if next_travel_event_car.time < t:
                    print('Problem 1')
                fes.add(next_travel_event_car)
                road = (car.path[car.progress -1], car.path[car.progress])
                if Graph.edges[road]['queue'] or Graph.edges[road]['accident']:
                    Queues[road].add_list([car], event.time)
                    LIST_AMOUNT_CARS_TRAFFIC_CHANGE.append(1)
                else:
                    LIST_AMOUNT_CARS_TRAFFIC_CHANGE.append(0)
                    
                
            
            # else:
                # print(event.car)

        #Start accident
        if event.type == 2:
            road = event.road
            #choose random cars affected:
                #List comprehension is inefficient af here
                # cars_not_in_queue = [car for car in DIC_EDGES[road] if car not in Queues[road].cars]
            cars_not_in_queue = sorted(list(set(DIC_EDGES[road]) - set(Queues[road].cars)))
            if len(cars_not_in_queue) != 0:
                amount_cars_affected = int(np.floor(event.location * len(cars_not_in_queue)))
                LIST_AMOUNT_CARS_TRAFFIC_CHANGE.append(amount_cars_affected)
                cars_affected = cars_not_in_queue[:amount_cars_affected]

                # print(f'Time enter queue {event.time}')
                Queues[road].add_list(cars_affected, event.time)
                
            Graph.edges[road]['accident'] = True
            #Assuming end of accident event is already created, else create here
            # print(f'{t}')

        #End accident
        if event.type == 3:
            road = event.road
            #1 minute until first car leaves the queue 
            if not(Queues[road].isEmpty()) and not(Graph.edges[road]['queue']):
                # print(Queues[road])
                # print(f'First event departure {Queues[road].isEmpty()}', road)
                first_departure_event = Event(4, event.time, road=road)
                # print(first_departure_event)
                fes.add(first_departure_event)
                if first_departure_event.time < t:
                    print('Problem 3')
                Graph.edges[road]['queue'] = True
            Graph.edges[road]['accident'] = False   
            # print(t)
            LIST_AMOUNT_CARS_TRAFFIC_CHANGE.append(0)
        # 
        #Departure from queue
        if event.type == 4:
            
            # print(event)
            road = event.road
            time = event.time
            #Remove car from queue
            # print(len(Queues[road].cars), event.road)
            car = Queues[road].next()
            LIST_AMOUNT_CARS_TRAFFIC_CHANGE.append(-1)
            # print(car.progress)
            #Modify time to arrival of next node in path
            # print(f'Time exit queue {event.time}')
            # print(f'Previous event at {car.time}')
            new_event_exit = car.exit_queue(time)
            # print(f'New event at {new_event_exit.time}')
            fes.add(new_event_exit)
            # print(f'Equivalent cars {car1==car}')
            # print(car in DIC_EDGES[(car.path[car.progress - 1], car.path[car.progress])])
            # heapq.heapify(fes.events)

            if not(Queues[road].isEmpty()):
                #Next departure event
                next_event_time = time + Graph.edges[road]['DepDist'].rvs()
                
                next_event = Event(4, next_event_time, road=road)
                if next_event.time < t:
                    print('Problem 4')
                # print(next_event)
                fes.add(next_event)
            else:
                Graph.edges[road]['queue'] = False

        # heapq.heapify(fes.events)
        # print(t)
        step += 1

In [ ]:
#Checking if cars make it to destionation
for i in range(0, len(LIST_CARS)):
    car_i = LIST_CARS[i]
    # if car_i.progress != len(car_i.path) - 1:
    #     print(f'oh oh {LIST_CARS[i].time}')
    if car_i.path[car_i.progress] != car_i.destination:
        print(f'oh oh {LIST_CARS[i].time}')

oh oh 24.095614776725863
oh oh 24.093644808391428
oh oh 24.061710558344625
oh oh 24.13674358823255
oh oh 24.072056754415804
oh oh 24.004859652011252
oh oh 24.10801473652794
oh oh 24.103557013092804
oh oh 24.158085818917762
oh oh 24.11152897022872
oh oh 24.135254991293312
oh oh 24.024747604613484
oh oh 24.245248118315345
oh oh 24.037362210920392
oh oh 24.036099913263527
oh oh 24.019381899956414
oh oh 24.233140397896637
oh oh 24.23338416623372
oh oh 24.08582472982564
oh oh 24.221101865242037
oh oh 24.097336539276156
oh oh 24.034500245308315
oh oh 24.039638260869726
oh oh 24.10344594478678
oh oh 24.063016625993296
oh oh 24.28686072005727
oh oh 24.001319222325865
oh oh 24.066794796319172
oh oh 24.13977864484308
oh oh 24.030423291334067
oh oh 24.040150103643338
oh oh 24.079718597432606
oh oh 24.19194551144926
oh oh 24.011925872003676
oh oh 24.045851580183143
oh oh 24.060985528233598
oh oh 24.05218510274011
oh oh 24.008823569943864
oh oh 24.009909160371034
oh oh 24.044431515877516
oh oh 24.0

In [85]:
#Proof that updating the car event also updates the one in the list
car_debug = Car(0)
list_debug = [car_debug.next_event]
print(list_debug[0].time)
car_debug.enter_queue(1)
car_debug.exit_queue(2)
print(list_debug[0].time)

AttributeError: 'NoneType' object has no attribute 'time'